In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
base = '/content/drive/MyDrive/proyecto_reciclaje'
for carpeta in ['organico', 'reciclable', 'no_reciclable']:
    ruta = os.path.join(base, carpeta)
    print(carpeta, '→', len(os.listdir(ruta)), 'fotos')

organico → 15 fotos
reciclable → 15 fotos
no_reciclable → 15 fotos


In [3]:
import tensorflow as tf

base = '/content/drive/MyDrive/proyecto_reciclaje'
IMG_SIZE = 160

dataset = tf.keras.utils.image_dataset_from_directory(
    base,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=16,
    seed=42
)
print('Clases:', dataset.class_names)

Found 45 files belonging to 3 classes.
Clases: ['no_reciclable', 'organico', 'reciclable']


In [4]:
from tensorflow.keras.applications import mobilenet_v2

def preparar(imagen, etiqueta):
    return mobilenet_v2.preprocess_input(imagen), etiqueta

ds = dataset.map(preparar)

In [5]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

modelo = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(3, activation='softmax')  # CAMBIO 1: 3 clases, no 1
])

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',  # CAMBIO 2: ya no es binario
    metrics=['accuracy']
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [6]:
historia = modelo.fit(ds, epochs=5)

Epoch 1/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.3333 - loss: 1.4796
Epoch 2/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 491ms/step - accuracy: 0.3111 - loss: 1.4719
Epoch 3/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 262ms/step - accuracy: 0.2667 - loss: 1.4058
Epoch 4/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 275ms/step - accuracy: 0.3556 - loss: 1.3964
Epoch 5/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 236ms/step - accuracy: 0.3556 - loss: 1.2730


In [7]:
modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

historia = modelo.fit(ds, epochs=15)

Epoch 1/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 365ms/step - accuracy: 0.3333 - loss: 1.3445
Epoch 2/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 424ms/step - accuracy: 0.6222 - loss: 0.9030
Epoch 3/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 260ms/step - accuracy: 0.7111 - loss: 0.7820
Epoch 4/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 247ms/step - accuracy: 0.8667 - loss: 0.5545
Epoch 5/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 237ms/step - accuracy: 0.9778 - loss: 0.3318
Epoch 6/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 270ms/step - accuracy: 0.9333 - loss: 0.2996
Epoch 7/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 271ms/step - accuracy: 0.9778 - loss: 0.2033
Epoch 8/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 305ms/step - accuracy: 0.9333 - loss: 0.2367
Epoch 9/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 256ms/step - accuracy: 0.9778 - loss: 0.1712
Epoch 10/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 397ms/step - accuracy: 1.0000 - loss: 0.1044
Epoch 11/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 400ms/step - accuracy: 1.0000 - loss: 0.0967
Epoch 12/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 442ms/step - accuracy: 1.0000 - lo

In [8]:
dataset_train = tf.keras.utils.image_dataset_from_directory(
    base, validation_split=0.2, subset='training', seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=16)

dataset_val = tf.keras.utils.image_dataset_from_directory(
    base, validation_split=0.2, subset='validation', seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=16)

Found 45 files belonging to 3 classes.
Using 36 files for training.
Found 45 files belonging to 3 classes.
Using 9 files for validation.


In [9]:
ds_train = dataset_train.map(preparar)
ds_val = dataset_val.map(preparar)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

modelo = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(3, activation='softmax')
])

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

historia = modelo.fit(ds_train, epochs=15)

print('--- EXAMEN FINAL (validación) ---')
resultado = modelo.evaluate(ds_val)
print('Accuracy real:', round(resultado[1]*100, 1), '%')

Epoch 1/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 305ms/step - accuracy: 0.4444 - loss: 1.2075
Epoch 2/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 192ms/step - accuracy: 0.7222 - loss: 0.8015
Epoch 3/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 174ms/step - accuracy: 0.8056 - loss: 0.5945
Epoch 4/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 190ms/step - accuracy: 0.9167 - loss: 0.4464
Epoch 5/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 175ms/step - accuracy: 0.9167 - loss: 0.3447
Epoch 6/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 185ms/step - accuracy: 1.0000 - loss: 0.2301
Epoch 7/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 176ms/step - accuracy: 0.9722 - loss: 0.2065
Epoch 8/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 176ms/step - accuracy: 1.0000 - loss: 0.1959
Epoch 9/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 170ms/step - accuracy: 1.0000 - loss: 0.1205
Epoch 10/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 165ms/step - accuracy: 1.0000 - loss: 0.1009
Epoch 11/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 308ms/step - accuracy: 1.0000 - loss: 0.0742
Epoch 12/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 292ms/step - accuracy: 1.0000 - lo

In [10]:
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

modelo = tf.keras.Sequential([
    data_aug,
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3, activation='softmax')
])

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

historia = modelo.fit(ds_train, epochs=25)

print('--- EXAMEN FINAL (validación) ---')
resultado = modelo.evaluate(ds_val)
print('Accuracy real:', round(resultado[1]*100, 1), '%')

Epoch 1/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 338ms/step - accuracy: 0.3333 - loss: 1.3891
Epoch 2/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 382ms/step - accuracy: 0.4167 - loss: 1.3040
Epoch 3/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 321ms/step - accuracy: 0.6667 - loss: 0.8410
Epoch 4/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 211ms/step - accuracy: 0.5833 - loss: 0.8397
Epoch 5/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 220ms/step - accuracy: 0.4444 - loss: 0.8550
Epoch 6/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 226ms/step - accuracy: 0.7500 - loss: 0.5502
Epoch 7/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 219ms/step - accuracy: 0.7500 - loss: 0.6048
Epoch 8/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 206ms/step - accuracy: 0.7222 - loss: 0.4803
Epoch 9/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 218ms/step - accuracy: 0.8333 - loss: 0.4170
Epoch 10/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 212ms/step - accuracy: 0.8611 - loss: 0.3583
Epoch 11/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 387ms/step - accuracy: 0.8611 - loss: 0.3004
Epoch 12/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 386ms/step - accuracy: 0.8889 - lo

In [12]:
import gradio as gr

clases = dataset_train.class_names

def clasificar(img):
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = mobilenet_v2.preprocess_input(img)
    img = tf.expand_dims(img, 0)
    pred = modelo.predict(img, verbose=0)[0]
    return {clases[i]: float(pred[i]) for i in range(len(clases))}

demo = gr.Interface(
    fn=clasificar,
    inputs=gr.Image(sources=['webcam'], type='numpy'),
    outputs=gr.Label(),
    title='Clasificador de residuos en vivo'
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://efc98f76574dd2fba6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),

])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

modelo = tf.keras.Sequential([
    data_aug,
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3, activation='softmax')
])

modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

historia = modelo.fit(ds_train, epochs=25)
resultado = modelo.evaluate(ds_val)
print('Accuracy real:', round(resultado[1]*100, 1), '%')

Epoch 1/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 239ms/step - accuracy: 0.3056 - loss: 1.3566
Epoch 2/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 222ms/step - accuracy: 0.3333 - loss: 1.4955
Epoch 3/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 227ms/step - accuracy: 0.3611 - loss: 1.2920
Epoch 4/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 234ms/step - accuracy: 0.4444 - loss: 1.1225
Epoch 5/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 230ms/step - accuracy: 0.7222 - loss: 0.6785
Epoch 6/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 224ms/step - accuracy: 0.7500 - loss: 0.6160
Epoch 7/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 407ms/step - accuracy: 0.8611 - loss: 0.4625
Epoch 8/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 414ms/step - accuracy: 0.7778 - loss: 0.6561
Epoch 9/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 324ms/step - accuracy: 0.8611 - loss: 0.4480
Epoch 10/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 229ms/step - accuracy: 0.9167 - loss: 0.3804
Epoch 11/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 226ms/step - accuracy: 0.9167 - loss: 0.3194
Epoch 12/25
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 227ms/step - accuracy: 0.9722 - lo